# arXiv Computer Science 최근 3개월 논문 수집

arXiv API에서 Computer Science 하위 카테고리 중 Artificial Intelligence(cs.AI), Machine Learning(cs.LG), Computation and Language(cs.CL), Computer Vision and Pattern Recognition(cs.CV), Information Retrieval(cs.IR), Robotics(cs.RO)에 해당하고, 노트북 실행일 기준 최근 3개월 이내에 제출된 논문을 조회해 JSONL 파일로 저장합니다.

> arXiv API 이용 시 요청 사이에 일정 간격을 두며, 기본 수집 상한은 `MAX_RESULTS`입니다. `CHUNK_SIZE`(기본 5,000)건마다 새로운 JSONL 파일(`arxiv_cs_recent_3months_part{n}.jsonl`)로 나누어 저장합니다.
>
> **arXiv API 한계:** `start`(조회 시작 위치)가 10,000 이상이면 API가 항상 HTTP 500을 반환합니다(일시적 오류가 아니라 API 자체의 하드 리밋). 이를 피하기 위해 전체 기간을 하위 날짜 구간으로 재귀적으로 쪼개서, 각 구간의 결과 수가 안전 상한(`SAFE_PAGE_LIMIT`, 기본 9,000)을 넘지 않도록 한 뒤 구간별로 따로 조회합니다.
>
> 수집 도중 에러가 나서 셀을 다시 실행하면, 같은 검색 조건(기간)일 경우 이전에 저장된 파일과 진행 상태(`arxiv_cs_recent_3months_state.json`)를 읽어 이미 완료된 날짜 구간은 건너뛰고 이어서 수집합니다. 날짜가 바뀌어 검색 조건(최근 3개월 범위)이 달라지면 기존 파일을 정리하고 처음부터 새로 수집합니다.

In [ ]:
from calendar import monthrange
from datetime import date, timedelta
from pathlib import Path
import http.client
import json
import ssl
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

MONTHS = 3
MAX_RESULTS = 1000000
PAGE_SIZE = 1000  # 공식 API 페이지 상한(2,000) 이내로 요청 횟수를 줄임
CHUNK_SIZE = 5000
SAFE_PAGE_LIMIT = 9000  # arXiv API는 start가 10,000 이상이면 항상 500 에러를 반환하므로 여유를 둔 안전 상한
# arXiv 권장(3초)보다 넉넉히 뒀던 20초로도 429가 계속 나서 30초로 더 늘렸습니다.
# (참고문헌 수집 노트북에서도 같은 API로 같은 문제를 겪었습니다: arxiv_references_to_jsonl.ipynb)
REQUEST_INTERVAL_SECONDS = 30.0
MAX_RETRIES = 15  # 429가 몰아치는 구간을 버틸 수 있도록 재시도 횟수를 늘림 (10 -> 15)
BACKOFF_BASE_SECONDS = 10.0
BACKOFF_MAX_SECONDS = 300.0
API_URL = 'https://export.arxiv.org/api/query'

# 수집 대상 카테고리: Artificial Intelligence, Machine Learning, Computation and Language,
# Computer Vision and Pattern Recognition, Information Retrieval, Robotics
CATEGORIES = ['cs.AI', 'cs.LG', 'cs.CL', 'cs.CV', 'cs.IR', 'cs.RO']
CATEGORY_QUERY = '(' + ' OR '.join(f'cat:{category}' for category in CATEGORIES) + ')'

def subtract_months(day, months):
    """월말에서도 안전하게 지정한 개월 수를 뺍니다."""
    month_index = day.year * 12 + day.month - 1 - months
    year, month_zero_based = divmod(month_index, 12)
    month = month_zero_based + 1
    return date(year, month, min(day.day, monthrange(year, month)[1]))

END_DATE = date.today()
START_DATE = subtract_months(END_DATE, MONTHS)
DATE_QUERY = f'submittedDate:[{START_DATE:%Y%m%d}0000 TO {END_DATE:%Y%m%d}2359]'
SEARCH_QUERY = f'{CATEGORY_QUERY} AND {DATE_QUERY}'

def range_query(range_start, range_end):
    return f'{CATEGORY_QUERY} AND submittedDate:[{range_start:%Y%m%d}0000 TO {range_end:%Y%m%d}2359]'

# 저장소 루트와 notebooks/ 어느 위치에서 커널을 시작해도 동작합니다.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 폴더에서 커널을 시작하세요.")

# data/ai는 arxiv_computer_science_recent_3months_to_jsonl_oai_pmh.ipynb가 cs.AI만
# 골라 저장하는 폴더라, 여기서 6개 분야를 섞어 넣으면 그 가정에 기대는 다른 노트북들
# (인용/참고문헌 수집)이 오염됩니다. 그래서 기존에 이 6개 분야 데이터가 이미 들어있는
# data/ai_5에 저장합니다.
OUTPUT_DIR = ROOT / 'data' / 'ai_5'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PREFIX = 'arxiv_cs_recent_3months'
STATE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_state.json'

def chunk_path(chunk_index):
    return OUTPUT_DIR / f'{FILE_PREFIX}_part{chunk_index}.jsonl'

print(f'조회 기간: {START_DATE} ~ {END_DATE}')
print(f'수집 카테고리: {", ".join(CATEGORIES)}')
print(f'검색식: {SEARCH_QUERY}')
print(f'저장 위치: {OUTPUT_DIR.resolve()} ({FILE_PREFIX}_part{{n}}.jsonl, {CHUNK_SIZE}건/파일)')
print(f'진행 상태 파일: {STATE_PATH.name} (재실행 시 이어받기에 사용)')

In [2]:
ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom'}
ARXIV_NS = {'arxiv': 'http://arxiv.org/schemas/atom'}
OPENSEARCH_NS = {'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'}

def text_or_none(parent, path, namespaces=ATOM_NS):
    node = parent.find(path, namespaces)
    return node.text.strip() if node is not None and node.text else None

def parse_entry(entry):
    authors = []
    for author in entry.findall('atom:author', ATOM_NS):
        name = text_or_none(author, 'atom:name')
        if name:
            authors.append(name)
    categories = [
        node.attrib['term']
        for node in entry.findall('atom:category', ATOM_NS)
        if 'term' in node.attrib
    ]
    links = {
        link.attrib.get('rel', 'alternate'): link.attrib.get('href')
        for link in entry.findall('atom:link', ATOM_NS)
    }
    published = text_or_none(entry, 'atom:published')
    primary_category_node = entry.find('arxiv:primary_category', ARXIV_NS)
    primary_category = primary_category_node.attrib.get('term') if primary_category_node is not None else None
    return {
        'id': text_or_none(entry, 'atom:id'),
        'title': ' '.join((text_or_none(entry, 'atom:title') or '').split()),
        'abstract': ' '.join((text_or_none(entry, 'atom:summary') or '').split()),
        'authors': authors,
        'categories': categories,
        'primary_category': primary_category,
        'published': published,
        'updated': text_or_none(entry, 'atom:updated'),
        'doi': text_or_none(entry, 'arxiv:doi', ARXIV_NS),
        'pdf_url': links.get('related') or links.get('alternate'),
        'source': 'arxiv',
        'collection_window': {
            'start': START_DATE.isoformat(),
            'end': END_DATE.isoformat(),
        },
    }

def _retry_delay(error, attempt):
    retry_after = getattr(error, 'headers', None) and error.headers.get('Retry-After')
    if retry_after:
        try:
            return max(float(retry_after), REQUEST_INTERVAL_SECONDS)
        except ValueError:
            pass
    return min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)

_last_request_at = 0.0
RETRYABLE_HTTP_CODES = {429, 500, 502, 503, 504}
RETRYABLE_NETWORK_ERRORS = (urllib.error.URLError, TimeoutError, ConnectionError, http.client.HTTPException)

def _urlopen_with_retry(request, timeout):
    global _last_request_at
    for attempt in range(MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        try:
            _last_request_at = time.monotonic()
            return urllib.request.urlopen(request, timeout=timeout, context=ssl.create_default_context())
        except urllib.error.HTTPError as error:
            if error.code not in RETRYABLE_HTTP_CODES or attempt >= MAX_RETRIES:
                raise
            delay = _retry_delay(error, attempt)
            print(f'HTTP {error.code}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            error.close()
            time.sleep(delay)
        except RETRYABLE_NETWORK_ERRORS as error:
            # 타임아웃/연결 끊김 등 응답 자체를 못 받은 경우도 재시도 대상에 포함
            if attempt >= MAX_RETRIES:
                raise
            delay = min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)
            print(f'네트워크 오류({error!r}): {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            time.sleep(delay)

def fetch_feed(query, start=0, max_results=PAGE_SIZE):
    """검색식으로 한 페이지를 조회하고 (전체 결과 수, 이 페이지의 논문 목록)을 반환합니다."""
    params = {
        'search_query': query,
        'start': start,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending',
    }
    url = f'{API_URL}?{urllib.parse.urlencode(params)}'
    request = urllib.request.Request(url, headers={'User-Agent': 'arxiv-cs-jsonl/1.0'})
    with _urlopen_with_retry(request, timeout=60) as response:
        root = ET.fromstring(response.read())
    total_node = root.find('opensearch:totalResults', OPENSEARCH_NS)
    total_results = int(total_node.text) if total_node is not None and total_node.text else 0
    entries = [parse_entry(entry) for entry in root.findall('atom:entry', ATOM_NS)]
    return total_results, entries

In [ ]:
def write_chunk(chunk_index, chunk_papers):
    path = chunk_path(chunk_index)
    with path.open('w', encoding='utf-8') as file:
        for paper in chunk_papers:
            file.write(json.dumps(paper, ensure_ascii=False) + '\n')
    return path

def load_state():
    if STATE_PATH.exists():
        with STATE_PATH.open(encoding='utf-8') as file:
            return json.load(file)
    return None

def save_state(completed_through):
    with STATE_PATH.open('w', encoding='utf-8') as file:
        json.dump({'search_query': SEARCH_QUERY, 'completed_through': completed_through.isoformat()}, file, ensure_ascii=False)

def load_existing_chunks():
    loaded = []
    chunk_index = 1
    while chunk_path(chunk_index).exists():
        with chunk_path(chunk_index).open(encoding='utf-8') as file:
            for line in file:
                if not line.strip():
                    continue
                try:
                    loaded.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f'  경고: {chunk_path(chunk_index).name}의 손상된 줄 하나를 건너뜁니다.')
        chunk_index += 1
    return loaded

def clear_previous_output():
    for old_file in OUTPUT_DIR.glob(f'{FILE_PREFIX}_part*.jsonl'):
        old_file.unlink()
    if STATE_PATH.exists():
        STATE_PATH.unlink()

def build_sub_ranges(range_start, range_end):
    """arXiv API는 start가 10,000 이상이면 항상 500 에러를 반환하므로,
    결과 수가 SAFE_PAGE_LIMIT을 넘는 구간은 날짜 중간점을 기준으로 재귀적으로 쪼갭니다."""
    total, _ = fetch_feed(range_query(range_start, range_end), start=0, max_results=1)
    if total > SAFE_PAGE_LIMIT and range_start < range_end:
        mid = range_start + (range_end - range_start) // 2
        return build_sub_ranges(range_start, mid) + build_sub_ranges(mid + timedelta(days=1), range_end)
    return [(range_start, range_end, total)]

print('전체 기간을 안전한 단위로 분할하는 중...')
sub_ranges = build_sub_ranges(START_DATE, END_DATE)
print(f'분할된 구간 수: {len(sub_ranges)}, 예상 총 논문 수: {sum(total for _, _, total in sub_ranges)}')
for range_start, range_end, total in sub_ranges:
    print(f'  {range_start} ~ {range_end}: {total}건')

state = load_state()
completed_through = None
if state and state.get('search_query') == SEARCH_QUERY:
    papers = load_existing_chunks()
    seen_ids = {paper['id'] for paper in papers if paper['id']}
    if state.get('completed_through'):
        completed_through = date.fromisoformat(state['completed_through'])
    print(f'이전 수집 이어받기: 기존 {len(papers)}개 논문 로드, {completed_through} 이후 구간부터 재개')
else:
    if state is not None:
        print('검색 조건(기간)이 이전 실행과 달라 기존 파일을 정리하고 처음부터 새로 수집합니다.')
    clear_previous_output()
    papers = []
    seen_ids = set()

next_chunk_index = 1

def flush_progress():
    global next_chunk_index
    completed_chunks = len(papers) // CHUNK_SIZE
    while next_chunk_index <= completed_chunks:
        chunk_start = (next_chunk_index - 1) * CHUNK_SIZE
        chunk_end = next_chunk_index * CHUNK_SIZE
        saved_path = write_chunk(next_chunk_index, papers[chunk_start:chunk_end])
        print(f'  -> {saved_path.name} 저장 ({chunk_end - chunk_start}건)')
        next_chunk_index += 1
    remainder_start = (next_chunk_index - 1) * CHUNK_SIZE
    if remainder_start < len(papers):
        write_chunk(next_chunk_index, papers[remainder_start:])

STAGNANT_PAGE_LIMIT = 5  # 새 논문이 하나도 없는 페이지가 이만큼 연속되면 이 구간은 다 받은 것으로 보고 조기 종료
# (arXiv API가 날짜 범위 페이지네이션에서 같은 페이지를 중복 반환하는 경우가 있어, 불필요한 API 호출을 막기 위함)

hit_cap = False
for range_start, range_end, total in sub_ranges:
    if completed_through is not None and range_end <= completed_through:
        continue  # 이미 완료된 구간은 건너뜀
    query = range_query(range_start, range_end)
    print(f'구간 수집 중: {range_start} ~ {range_end} (예상 {total}건)')
    stagnant_pages = 0
    for start in range(0, total, PAGE_SIZE):
        _, page = fetch_feed(query, start=start, max_results=min(PAGE_SIZE, total - start))
        before_count = len(papers)
        for paper in page:
            if paper['id'] and paper['id'] not in seen_ids:
                seen_ids.add(paper['id'])
                papers.append(paper)
        stagnant_pages = stagnant_pages + 1 if len(page) > 0 and len(papers) == before_count else 0
        if stagnant_pages > 0:
            print(f'  {len(papers)}개 누적 수집 (신규 없음 {stagnant_pages}/{STAGNANT_PAGE_LIMIT} - {STAGNANT_PAGE_LIMIT}에 도달하면 이 구간을 자동으로 넘어갑니다)')
        else:
            print(f'  {len(papers)}개 누적 수집')
        flush_progress()
        if len(papers) >= MAX_RESULTS:
            hit_cap = True
            break
        if len(page) == 0:
            break
        if stagnant_pages >= STAGNANT_PAGE_LIMIT:
            print(f'  경고: 새 논문 없는 페이지가 {STAGNANT_PAGE_LIMIT}번 연속 발생해 이 구간을 조기 종료합니다.')
            break
    if hit_cap:
        break
    save_state(range_end)  # 이 구간까지는 완전히 수집됨

if not hit_cap and STATE_PATH.exists():
    STATE_PATH.unlink()  # 모든 구간이 정상적으로 끝까지 완료됐으므로 진행 상태 파일은 정리

papers = papers[:MAX_RESULTS]
total_chunks = max(1, (len(papers) + CHUNK_SIZE - 1) // CHUNK_SIZE)
for chunk_index in range(1, total_chunks + 1):
    chunk_start = (chunk_index - 1) * CHUNK_SIZE
    chunk_end = min(chunk_index * CHUNK_SIZE, len(papers))
    write_chunk(chunk_index, papers[chunk_start:chunk_end])

print(f'완료: {len(papers)}개 논문을 {total_chunks}개 파일({FILE_PREFIX}_part1~{total_chunks}.jsonl, 최대 {CHUNK_SIZE}건/파일)로 저장했습니다.')

전체 기간을 안전한 단위로 분할하는 중...
HTTP 429: 10초 후 재시도 (1/10)
HTTP 429: 20초 후 재시도 (2/10)


In [ ]:
# JSONL 저장 결과 및 기간 조건 검증 (분할된 모든 파일을 합쳐서 확인)
saved_papers = []
for chunk_index in range(1, total_chunks + 1):
    with chunk_path(chunk_index).open(encoding='utf-8') as file:
        saved_papers.extend(json.loads(line) for line in file if line.strip())

assert len(saved_papers) == len(papers)
assert len({paper['id'] for paper in saved_papers}) == len(saved_papers)
assert all(paper['id'] and paper['title'] and paper['abstract'] for paper in saved_papers)
assert all(START_DATE.isoformat() <= paper['published'][:10] <= END_DATE.isoformat() for paper in saved_papers)
print(f'검증 완료: {len(saved_papers)}개 레코드, {total_chunks}개 파일, 중복 없음, 최근 3개월 범위 확인')
display(saved_papers[:3])

검증 완료: 38479개 레코드, 8개 파일, 중복 없음, 최근 3개월 범위 확인


[{'id': 'http://arxiv.org/abs/2609.11929v1',
  'title': 'SenseNova-U1.5: Towards Native Unified Visual Intelligence',
  'abstract': 'We launch SenseNova-U1.5, an 8B-MoT native unified multimodal model that understands, reasons about, and generates visual content within an encoder-free and VAE-free architecture. We strengthen its visual interface through spatially coherent patch reconstruction and scale its training with carefully curated generation and editing data, improved task formulation, structural prompt enhancement, and native resolutions of up to 4K. For post-training, we optimize specialized experts for visual aesthetics, bilingual text rendering, infographic generation, and image editing, and consolidate their capabilities through multi-expert on-policy distillation. Across extensive evaluations, SenseNova-U1.5 largely advances image fidelity, text rendering, complex composition, multi-reference editing, and interleaved generation, while improving instruction following and pr